# 03 - Pharmaceutical DPO

Public pharmaceutical preference data + Qwen2.5-0.5B + LoRA + DPO. Teaching experiment only.

In [ ]:
!pip -q install -U "transformers>=4.55,<5" "datasets>=3.6,<5" "peft>=0.17,<1" "trl>=0.21,<1" "accelerate>=1.10,<2" "bitsandbytes>=0.46,<1" "torchao>=0.16,<1"

In [ ]:
import torch
import transformers, datasets, peft, trl
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Colab.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from datasets import load_dataset, Dataset
RAW_DATASET='ThakrePranjal/pharma-preference-dataset'
raw=load_dataset(RAW_DATASET, split='train')
print(raw)
print(raw.column_names)
required={'prompt','chosen','rejected'}
missing=required.difference(raw.column_names)
if missing: raise ValueError(f'Missing columns: {sorted(missing)}')

In [ ]:
def clean_prompt(text):
    text=str(text).strip()
    if '### Instruction:' in text: text=text.split('### Instruction:',1)[1]
    if '### Response:' in text: text=text.split('### Response:',1)[0]
    return text.strip()
rows=[]
for row in raw:
    rows.append({'prompt':[{'role':'user','content':clean_prompt(row['prompt'])}], 'chosen':[{'role':'assistant','content':str(row['chosen']).strip()}], 'rejected':[{'role':'assistant','content':str(row['rejected']).strip()}]})
dataset=Dataset.from_list(rows)
split=dataset.train_test_split(test_size=0.2,seed=42)
train_dataset=split['train']
eval_dataset=split['test']
print('Total:',len(dataset),'Train:',len(train_dataset),'Eval:',len(eval_dataset))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL_NAME='Qwen/Qwen2.5-0.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='left'
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=dtype).cuda()
print('Has chat template:',tokenizer.chat_template is not None)

In [ ]:
def generate(model, question, max_new_tokens=120):
    messages=[{'role':'user','content':question}]
    encoded=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True)
    encoded={k:v.to(model.device) for k,v in encoded.items()}
    with torch.no_grad():
        output=model.generate(**encoded,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(output[0],skip_special_tokens=True)
test_prompt=eval_dataset[0]['prompt'][0]['content']
print('QUESTION:',test_prompt)
print()
print('BEFORE DPO:')
print(generate(model,test_prompt))

In [ ]:
from peft import LoraConfig
from trl import DPOConfig,DPOTrainer
lora_config=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,target_modules=['q_proj','k_proj','v_proj','o_proj'],bias='none',task_type='CAUSAL_LM')
dpo_args=DPOConfig(output_dir='./outputs/pharma-dpo',per_device_train_batch_size=2,gradient_accumulation_steps=4,num_train_epochs=5,learning_rate=1e-5,beta=0.1,max_length=512,logging_steps=2,save_strategy='no',report_to='none',fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported())
trainer=DPOTrainer(model=model,args=dpo_args,train_dataset=train_dataset,eval_dataset=eval_dataset,processing_class=tokenizer,peft_config=lora_config)
trainer.train()

In [ ]:
metrics=trainer.evaluate()
for key,value in metrics.items():
    print(f'{key}: {value:.4f}' if isinstance(value,(int,float)) else f'{key}: {value}')
model.eval()
print()
print('AFTER DPO:')
print(generate(model,test_prompt))

In [ ]:
ADAPTER_DIR='./outputs/pharma-dpo-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:',ADAPTER_DIR)